### Anwendungen der Klasse Vector
Ziel ist es, das Zifferblatt einer Uhr zu zeichnen.
Dann nutzen wir unsere Funktion `draw_arrow` um einen Uhrzeiger als Pfeil zu zeichnen,
den wir dann jeweils um 360/60 = 6 Grad rotieren wollen.

In [6]:
import math
from vector import Vector as Vec


center = (50, 50)
radius = 40
center_vec = Vec(*center)
hand = Vec(0, 1)
hand, center

(Vec(0, 1), (50, 50))

In [7]:
# 1-Minutenmarkierung
hand = Vec(0, 1)
hand = hand.rotate(math.pi/30)
start = (center_vec + 0.9*radius*hand).as_tuple()
end = (center_vec + 1.1*radius*hand).as_tuple()
start, end

((46.23697532236447, 85.80278823325784),
 (45.40074761622325, 93.75896339620402))

In [8]:
# 5-Minutenmarkierung
hand = Vec(0, 1)
hand = hand.rotate(math.pi/6)
start = (center_vec + 0.9*radius*hand).as_tuple()
end = (center_vec + 1.1*radius*hand).as_tuple()
start, end

((32.0, 81.17691453623979), (28.000000000000004, 88.1051177665153))

In [1]:
import math
from vector import Vector as Vec
from decorators import save_canvas_state


@save_canvas_state
def draw_arrow(canvas, start, end,
               tip_size=5, line_width=1,
               color='black', tip_color='red'):
    start = Vec(*start)
    end = Vec(*end)
    arrow = end - start
    alpha = math.atan2(arrow.y, arrow.x)

    tip = [Vec(*pt) for pt in ((-2, 1), (-2, -1), (0, 0))]
    rotated_tip = [v.rotate(alpha) for v in tip]
    translated_and_scaled_tip = [end + w*tip_size for w in rotated_tip]
    pts = [v.as_tuple() for v in translated_and_scaled_tip]

    canvas.line_width = line_width
    canvas.stroke_style = color
    canvas.fill_style = tip_color
    canvas.stroke_line(*start.as_tuple(), *end.as_tuple())
    canvas.fill_polygon(pts)

### Aufgabe
1. Schreibe eine Funktion `draw_clock(canvas, center, radius)`, die ein
Zifferblatt mit dem gegebenen Radius an Position center auf die Leinwand zeichnet.
2. Schreibe eine Funktion `draw_hand(canvas, center, radius, minute)`,
   die einen Pfeil von Zentrum der oben gezeichneten Uhr zur entsprechenden Minutenmarkierung zeichnet.

In [26]:
@save_canvas_state
def draw_clock(canvas, center, radius):
    canvas.stroke_style = 'black'
    canvas.line_width = 1
    canvas.stroke_circle(*center, radius)

    canvas.stroke_style = 'blue'
    canvas.line_width = 1
    center_vec = Vec(*center)
    hand = Vec(0, 1)
    for i in range(60):
        hand = hand.rotate(math.pi/30)
        start = (center_vec + 0.9*radius*hand).as_tuple()
        end = (center_vec + 1.1*radius*hand).as_tuple()
        canvas.stroke_line(*start, *end)

    canvas.line_width = 3
    for i in range(12):
        hand = hand.rotate(math.pi/6)
        start = (center_vec + 0.9*radius*hand).as_tuple()
        end = (center_vec + 1.1*radius*hand).as_tuple()
        canvas.stroke_line(*start, *end)


def draw_hand(canvas, center, radius, minute):
    hand = Vec(0, -1)
    center_vec = Vec(*center)
    hand = hand.rotate(minute*math.pi/30)

    tip_pos = (center_vec + .8*radius*hand).as_tuple()
    draw_arrow(canvas, center, tip_pos)

In [21]:
import widget_helpers as W

mcanvas = W.get_mcanvas(2)
fg, bg = mcanvas
mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [22]:
center = (50, 50)
radius = 40
draw_clock(bg, center, radius)

In [30]:
draw_hand(fg, center, radius, 13)

In [32]:
import time

for i in range(60):
    fg.clear()
    draw_hand(fg, center, radius, i)
    time.sleep(1)

In [33]:
import asyncio


def run_the_clock():
    async def animate():
        for i in range(60):
            fg.clear()
            draw_hand(fg, center, radius, i)
            await asyncio.sleep(1)

    task = asyncio.create_task(animate())
    return task

In [ ]:
task = run_the_clock()

In [125]:
task.cancel()

True